## Purpose of This Notebook

This notebook serves as an exploratory tool for examining the log files produced during the horse behavioural experiments conducted in October and November 2023. It facilitates the experimentation with text parsing techniques on the files before they are imported into a database. The primary objectives are:

1. To reconcile which log files should be included in or excluded from the analysis.
2. To conduct experiments with regular expressions ([`regex`](https://docs.python.org/3/howto/regex.html)) aimed at extracting pertinent data and fields from the log files.

## Experiment details and naming conventions

There are two main types of Experiment:

- Reward Prediction (RPE)
- Cognitive Bias (CB)

The RPE have the following subtypes:

- `RPE-A` : acquisition of response
- `RPE-H` : habit formation
- `RPE-E` : extinction of response
- `RPE-ER` : extinction prior to reinstatemnt of response
- `RPE-R` : reinstatement of response

### RPE-type experiments

A new experiment type was created during the experiments which was not in the original specification.
Can extract experiment details from Comment field. *TODO: CH to confirm details.*

*TODO: CH - I am also seeing an RPE-ER type?*

#### Logfile Exclusion rules

Logfiles that are from test runs and also bad data need to be excluded from the analysis.

Rules are **case-insensitive**. Files which satisfy the following conditions are excluded:

- All files with `_TEST_` as the subject name
- All files with `_FRECKLE_` as the subject name
- Some files with `_BONNIE_` (14 legitimate files - *TODO: CH to confirm details*)
- All files with `test` in the `Comment` field
- Possibly some logs with very short run-times - *TODO: CH to confirm the files that have been identified*
- All files with `_OLIVE_` as the subject name - probably discard, treat as optional for now (*TODO: CH to confirm treatment*)

#### Problems with log file names during experiments

- Appears that for greater than 20 trials (subjects?) the value in the filename was reported as `NaN`. e.g. see `pumba` experiments.
- Other `NaN`s? There are 31 files with `NaN` (some are restarts) - *TODO: Confirm with CH why were there restarts? Why not "new" experiment?*

#### Time differences

For each trial we calculate the following time differences: 

1. Time delta: (`touch` datetime - `start tone` datetime) 
2. Time delta: (`Next start` datetime - `dispense of pellets` datetime)
3. Time delta: (`Dispense final pellets` datetime - `start tone` datetime)

We use item 3 as cross check on the consistency of previous time deltas.

These calculated quantities are the same for all RFE-type experiments.

### Cognitive Bias Experiments

- Extract "RIGHT" or "LEFT" from the Comment field.
- Also extract details from log file name? *TODO: Confirm CH*
- 1. Training experiments: 
  - Type 1
  - Type 2
-  2. Testing experiments: 
   -  Type 1
   -  Type 2
   -  Type 3
   -  Type 4 (re-uses Type 1 with indicator to distinguish in Comment field)

#### Time differences

For each trial: 

Training Type 1 (randomised versus fixed):
- `Start` datetime = Green button pressed and horse is released
- Capture positive (`GO`) / negative (`NOGO`) response time subject to maximum cutoff time (e.g. 30 seconds)
- In addition to left/right positioning of feed, there are also median, near positive and near negative positions.

*TODO check with CH: Test only - be in all?*

## TODOs

1. Create lists of various filenames (include, exclude, etc) for CH to review. *DONE*
2. [TODO]

In [ ]:
# | echo: false

import inspect
import sys
from dataclasses import fields
from pathlib import Path
from pprint import pprint

import duckdb
import pandas as pd
from horse_logic.logfiles import Event, Experiment, Logfile, Logs, Parameter, Trial
from IPython.display import Markdown, display
from loguru import logger

In [ ]:
# | echo: false

def display_class_definition(class_name):
    display(Markdown(f"```{inspect.getsource(class_name)}```"))

In [ ]:
# | echo: false

import itables.options as opt
from itables import init_notebook_mode

init_notebook_mode(all_interactive=True)  # Display dataframes in a more friendly paginated manner
opt.pageLength = 20


In [ ]:
# | echo: false

def set_custom_logger_format():
    # custom_format = "<level>{level: <8}</level> | <cyan>{name}</cyan>:<cyan>{function}</cyan>:<cyan>{line}</cyan> - <level>{message}</level>"
    custom_format_no_func_info = "<level>{level: <8}</level> | <level>{message}</level>"
    logger.remove()    # Remove the default handler before adding a new one
    logger.add(sys.stderr, format=custom_format_no_func_info)

set_custom_logger_format()

In [ ]:
# | echo: false

# Custom notebook analysis functions

def display_sessions_by_subject(df_out, subject_df):
    for subject in subject_df["subject_name"]:
        df_ = df_out.drop(columns=["suffix", "subject_name"])[df_out["subject_name"] == subject]
        df_.loc[:, "time_dff"] = df_["datetime"].diff()
        display(Markdown(f"### {subject}: {len(df_)} session(s)"))
        display(df_)

### Load Subject order

In [ ]:
# | echo: false

subject_df = pd.read_csv("../data/subject_order.csv")

In [ ]:
# | echo: false

logger.info(f"Subjects: {len(subject_df)}")
logger.info(subject_df["subject_name"].to_list())

## Log file reconciliation

### Directory information: log file data and outputs

In [ ]:
# | echo: false

LOGFILES_DIR = "../data/results/zips/data_17Jan2020_email_hillydale_equine"

assert Path(LOGFILES_DIR).exists()
logger.info(f"Logfiles dir: {LOGFILES_DIR}")


In [ ]:
# | echo: false

OUTPUT_DIR = "../data/results/notebooks"

assert Path(OUTPUT_DIR).exists()
logger.info(f"Outputs dir: {OUTPUT_DIR}")

In [ ]:
# | echo: false

DATA_DIR = "../data"

assert Path(DATA_DIR).exists()

DATA_DB  = Path(DATA_DIR) / "experiments_October2023.ddb"  # DuckDB name
logger.info(f"Database file: {DATA_DB}")

### Initial exclusions (rule-based)

In [ ]:
# | echo: false

# Define the default set of log files to exclude before fine-tuning after
logs = Logs(path=LOGFILES_DIR, patterns=["*_test*.log", "*freckle*.log"], include=False)

In [ ]:
# | echo: false

logger.info(logs)
logger.info(f"Included logfiles: {len(logs.included_files)}")
logger.info(f"Excluded logfiles: {len(logs.excluded_files)}")

In [ ]:
# | echo: false

df_excluded = pd.DataFrame(logs.excluded_files, columns=["Excluded"])

In [ ]:
# | echo: false

df_excluded.sort_values(by="Excluded").reset_index(drop=True)

#### Included log files

In [ ]:
# | echo: false

df_included = pd.DataFrame(logs.included_files, columns=["Included"])

In [ ]:
# | echo: false

df_included.sort_values(by="Included").reset_index(drop=True).to_csv(Path(OUTPUT_DIR) / "included_files.csv", header=None)

In [ ]:
# | echo: false

df_included.sort_values(by="Included").reset_index(drop=True)

In [ ]:
# | echo: false

df_file_components = logs.create_filename_components_dataframe(logs.included_files)

In [ ]:
# | echo: false

df_out = df_file_components.sort_values(by=["subject_name", "session_number"]).reset_index(drop=True)

### Sessions summary by subject name

In [ ]:
# | echo: false

display_sessions_by_subject(df_out, subject_df)

In [ ]:
# | echo: false

df_out.drop(labels=["suffix"], axis=1, inplace=True)

In [ ]:
# | echo: false

df_out = df_out[["original_filename", "subject_name", "experiment_type", "session_number", "datetime"]]

In [ ]:
# | echo: false

df_out

In [ ]:
# | echo: false

# Calculate time differences between Experiments to look for anomalies

df_out.loc[:, "time_diff"] = df_out["datetime"].diff()

In [ ]:
# | echo: false

df_out

In [ ]:
# | echo: false

# Convert session number to integers - not sure how we got some floats?

df_out["session_number"] = df_out["session_number"].astype(pd.Int64Dtype())

In [ ]:
# | echo: false

# Get all Experiments where Bonnie was the subject
df_bonnie = df_out[df_out["subject_name"] == "bonnie"]

In [ ]:
# | echo: false

df_bonnie[df_bonnie["time_diff"] < pd.Timedelta("5m")]

In [ ]:
# | echo: false

bonnie_exclude_list = sorted(df_bonnie[df_bonnie["time_diff"] < pd.Timedelta("5m")]["original_filename"].tolist())

In [ ]:
# | echo: false

pd.DataFrame(bonnie_exclude_list, columns=["col1"]).sort_values(by="col1").reset_index(drop=True).to_csv(Path(OUTPUT_DIR) / "bonnie_excluded_files.csv", header=None)

In [ ]:
# | echo: false

len(bonnie_exclude_list)

In [ ]:
# | echo: false

logs.handle_specific_files(bonnie_exclude_list, include=False)

In [ ]:
# | echo: false

len(logs.excluded_files)

In [ ]:
# | echo: false

# Export list of all excluded files - by rules and specific exclusions

pd.DataFrame(logs.excluded_files, columns=["col1"]).sort_values(by="col1").reset_index(drop=True).to_csv(Path(OUTPUT_DIR) / "all_excluded_files.csv", header=None)

In [ ]:
# | echo: false

# !open $LOGFILES_DIR/Experiment_2023-10-01T21:30:25.370452_bonnie_1_RPE-ER.log

In [ ]:
# | echo: false

#!open $LOGFILES_DIR/Experiment_2023-10-02T21:29:38.930298_bonnie_3_RPE-E.log

In [ ]:
# | echo: false

# !open $LOGFILES_DIR/Experiment_2023-10-03T14:46:54.142393_bonnie_13_RPE-A.log

In [ ]:
# | echo: false

# !open $LOGFILES_DIR/Experiment_2023-10-19T14:37:56.973616_bonnie_72_RPE-H.log

In [ ]:
# | echo: false

df_out.to_csv(Path(OUTPUT_DIR) / "experiment_summary_Oct2023_included.csv")

### Explore `regex` parsing of a log file

Analysis code is in `logfiles.py`

Specific example defined below.

In [ ]:
# | echo: false

logfile_eg = f"{logs.path}/Experiment_2023-11-17T05:11:53.655051_clover_nan_RPE-R.log"

In [ ]:
# | echo: false

# Load example log file into Logfile class for processing
log_eg = Logfile(logfile_eg)

In [ ]:
# | echo: false

print(f"Example log file for regex parsing: {log_eg.file_name}")

#### Experiment table

In [ ]:
# | echo: false

display_class_definition(Experiment)

In [ ]:
# | echo: false

pprint(log_eg.parse_filename_components())

In [ ]:
# | echo: false

log_eg.parse_comments()

#### Parameters table

In [ ]:
# | echo: false

display_class_definition(Parameter)

In [ ]:
# | echo: false

log_eg.parse_parameters()

#### Trials and Events

In [ ]:
# | echo: false

display_class_definition(Trial)

In [ ]:
# | echo: false

display_class_definition(Event)

In [ ]:
# | echo: false

pprint(log_eg.parse_trials_and_events()[0])

In [ ]:
# | echo: false

n_trial = len(log_eg.parse_trials_and_events()[0])

In [ ]:
# | echo: false
 
df_trial_event = pd.DataFrame(log_eg.parse_trials_and_events()[1])

In [ ]:
# | echo: false

def display_events_by_trial_number(df_trial_event):
    n_trial = df_trial_event["TrialNumber"].max()
    for i_trial in range(n_trial):
        df_trial_event.loc[:, "time_diff"] = pd.to_datetime(df_trial_event["EventDateTime"]).diff()
        display(Markdown(f"### Trial #{i_trial+1}"))
        display(df_trial_event[df_trial_event["TrialNumber"] == i_trial+1])

In [ ]:

# | echo: false

def extract_events_by_trial_number(logfile):
    df_trial_event = pd.DataFrame(logfile.parse_trials_and_events()[1])
    n_trial = df_trial_event["TrialNumber"].max()
    for trial_id in range(n_trial):
        df_trial_event.loc[:, "time_diff"] = pd.to_datetime(df_trial_event["EventDateTime"]).diff()
        #display(Markdown(f"### Trial #{i_trial+1}"))
        #display(df_trial_event[df_trial_event["TrialNumber"] == i_trial+1])
        Trial(trial_id, exp_id, trial_number, start_tone_dt)

In [ ]:
# | echo: false

pd.DataFrame(log_eg.parse_trials_and_events()[1])

In [ ]:
# | echo: false

df_ = pd.DataFrame(log_eg.parse_trials_and_events()[1])

In [ ]:
# | echo: false

df_

In [ ]:
# | echo: false

# Test extract info for trials

event_id = 100

for i_trial in range(df_["TrialNumber"].max()):
    for row in df_[df_["TrialNumber"] == i_trial].itertuples():
        print(event_id, i_trial, row[2], row[3])
        #Event(event_id, i_trial, )

In [ ]:
# | echo: false

# df_trial_event.to_csv("df_trial_event_eg.csv")

In [ ]:
# | echo: false

display_events_by_trial_number(df_trial_event)

In [ ]:
# | echo: false

pprint(log_eg.parse_filename_components())

### Put data in DuckDB database

In [ ]:
def insert_record(con, record):
    table_name = record.__class__.__name__.lower() + 's'  # Assumes table name is plural of the class name
    columns = ', '.join(field.name for field in fields(record))
    placeholders = ', '.join('?' for _ in fields(record))
    values = tuple(getattr(record, field.name) for field in fields(record))
    query = f'INSERT INTO {table_name} ({columns}) VALUES ({placeholders})'
    con.execute(query, values)

In [ ]:
con = duckdb.connect(database=str(DATA_DB))

In [ ]:
def create_tables_from_sql_file(con, file_path):
    with open(file_path, 'r') as file:
        sql_commands = file.read()
    display(Markdown(f"\n```\n{sql_commands}\n```"))
    con.execute(sql_commands)

In [ ]:
create_tables_from_sql_file(con, '../sql/create_experiment_tables_ddb.sql')

In [ ]:
trial_id = 0
event_id = 0
# exp_id = 0 (this is handled in the outer loop)

for exp_id, logfile in enumerate(sorted(logs.included_files)): # Loop over Experiment
    print(f"{logs.path}/{logfile}")
    exp_details = Logfile(f"{logs.path}/{logfile}").parse_filename_components()
    exp_details.ExperimentID = exp_id
    insert_record(con, exp_details)

    # Loop over Trials

    trial_id += 1
    df_trial_event = pd.DataFrame(logfile.parse_trials_and_events()[1])
    n_trial = df_trial_event["TrialNumber"].max()
    for i_trial in range(n_trial):
        df_trial_event.loc[:, "time_diff"] = pd.to_datetime(df_trial_event["EventDateTime"]).diff()
        #display(Markdown(f"### Trial #{i_trial+1}"))
        #display(df_trial_event[df_trial_event["TrialNumber"] == i_trial+1])

        Trial(trial_id, exp_id, i_trial, start_tone_dt)    # Review database structure - what do we want to capture for the analysis

        # Loop over Events

        # TODO - Unpack events from DataFrame?



In [ ]:
con.sql("SHOW TABLES;")

In [ ]:
experiments_df = con.sql("SELECT * FROM Experiments").df()

In [ ]:
experiments_df

In [ ]:
con.sql("SELECT DISTINCT ExperimentType FROM Experiments")

In [ ]:
con.sql("SELECT DISTINCT SubjectName FROM Experiments ORDER BY SubjectName")